# Base-rate merged results

Load multi-model results from downloaded Kaggle Benchmarks runs, or from a local merged CSV.

Each row has **`score`** (`true`/`false`): whether the parsed answer matches **`scepticism_score_target`**. Unparseable rows have `score=false` and `parseable=false`.

**Kaggle (all evaluated models):** after `kaggle auth login`:

```bash
kaggle benchmarks tasks download base-rate-normative-accuracy \
  -o data/kaggle_runs/base-rate-normative-accuracy
```

Or run `python scripts/export_base_rate_kaggle_results.py --download`.

Set `LOAD_FROM_KAGGLE = True` in the next cell (default). Each aggregate `*.run.json` embeds per-prompt subruns from `base_rate_prompt_response`.

In [146]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "base_rate").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.build_base_rate_prompts import PROBLEM_TYPES
from benchmarks.kaggle_runs import (
    DEFAULT_BASE_RATE_TASK_SLUG,
    download_task_runs,
    find_run_json_files,
    merged_results_from_kaggle_runs,
)

LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = False  # True to refresh via Kaggle CLI before loading
KAGGLE_TASK_SLUG = DEFAULT_BASE_RATE_TASK_SLUG
KAGGLE_RUNS_DIR = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
BENCHMARK_CSV = ROOT / "data" / "base_rate" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "base_rate"

if LOAD_FROM_KAGGLE:
    if DOWNLOAD_KAGGLE_RUNS:
        download_task_runs(KAGGLE_TASK_SLUG, KAGGLE_RUNS_DIR)
    if not KAGGLE_RUNS_DIR.is_dir():
        raise FileNotFoundError(
            f"Download directory not found: {KAGGLE_RUNS_DIR}\n"
            f"Run: kaggle benchmarks tasks download {KAGGLE_TASK_SLUG} "
            f"-o {KAGGLE_RUNS_DIR}"
        )
    run_files = find_run_json_files(KAGGLE_RUNS_DIR)
    merged_rows = merged_results_from_kaggle_runs(
        KAGGLE_RUNS_DIR,
        benchmark_path=BENCHMARK_CSV,
    )
    df = pd.DataFrame(merged_rows)
    data_source = f"Kaggle runs ({len(run_files)} *.run.json under {KAGGLE_RUNS_DIR})"
else:
    merged_candidates = sorted(
        (
            candidate
            for candidate in MERGED_DIR.glob("base_rate_merged_results*.csv")
            if not candidate.name.endswith(".sav.csv")
        ),
        key=lambda candidate: candidate.stat().st_mtime,
        reverse=True,
    )
    if not merged_candidates:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run the base-rate benchmark notebook."
        )
    MERGED_CSV = merged_candidates[0]
    df = pd.read_csv(MERGED_CSV)
    data_source = str(MERGED_CSV)

if "score" in df.columns:
    df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
elif "score_outcome" in df.columns:
    df["score_value"] = (df["score_outcome"] == "normative").astype(int)
else:
    raise KeyError("Merged data must include 'score' or legacy 'score_outcome'.")

if "parseable" in df.columns:
    df["parseable_bool"] = df["parseable"].astype(str).str.lower().eq("true")

df["score_true"] = df["score_value"].astype(bool)

print("Source:", data_source)
print("Rows:", len(df))
print("Models:", sorted(df["model"].unique()))
print("Vignettes:", df["vignette_name"].nunique())
df.head()

Source: Kaggle runs (12 *.run.json under c:\src2\sceptical-llms\data\kaggle_runs\base-rate-normative-accuracy)
Rows: 264
Models: ['anthropic/claude-opus-4-1@20250805', 'anthropic/claude-opus-4-8@default', 'anthropic/claude-sonnet-4@20250514', 'google/gemini-3-flash-preview', 'google/gemini-3.5-flash', 'openai/gpt-5.5-2026-04-23']
Vignettes: 9


,example_id,vignette_name,problem_type,intersection_size,response_type,has_statistics,variant,prompt,well_posed,normative,...,parsed_answer_type,parsed_percent,parsed_choice,parsed_confidence,scoring_type,parseable,score,score_value,parseable_bool,score_true
0,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,implausible,0,mc_full,true,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered ...,true,implausible,...,unparseable,,,,mc_full,false,false,0,False,False
1,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,implausible,0,mc_full,true,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered ...,true,implausible,...,unparseable,,,,mc_full,false,false,0,False,False
2,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,implausible,0,mc_full,true,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered ...,true,implausible,...,mc_choice,,C,,mc_full,true,false,0,True,False
3,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,implausible,0,mc_full,true,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered ...,true,implausible,...,unparseable,,,,mc_full,false,false,0,False,False
4,ca_trump_voter__implausible__mc_full_probs,CA Trump voter,implausible,0,mc_full,true,mc_full_probs,You are a statistical consultant. Your task is to estimate a conditional probability from the information below.\n\nAmong US registered ...,true,implausible,...,mc_choice,,E,,mc_full,true,false,0,True,False


In [147]:
CANONICAL_PROBLEM_TYPE_ORDER = [
    "well_posed",
    "overlap_explicit",
    "overlap_implicit",
    "implausible",
]
if set(CANONICAL_PROBLEM_TYPE_ORDER) != PROBLEM_TYPES:
    raise ValueError(
        "CANONICAL_PROBLEM_TYPE_ORDER is out of sync with "
        "scripts.build_base_rate_prompts.PROBLEM_TYPES"
    )

observed_problem_types = list(df["problem_type"].dropna().unique())
legacy_problem_types = sorted(set(observed_problem_types) - PROBLEM_TYPES)
problem_type_order = [
    value
    for value in CANONICAL_PROBLEM_TYPE_ORDER
    if value in observed_problem_types
] + legacy_problem_types

if legacy_problem_types:
    print(
        "Legacy problem_type labels in merged file (pre explicit/implicit split):",
        legacy_problem_types,
    )

_work = df.copy()
if "parseable_bool" not in _work.columns:
    _work["parseable_bool"] = True
_work["score_miss"] = _work["parseable_bool"] & (_work["score_value"] == 0)
_work["unparseable_row"] = ~_work["parseable_bool"]

_grouped = _work.groupby("problem_type", observed=True)
results_by_problem_type = (
    pd.DataFrame(
        {
            "n": _grouped.size(),
            "score_true": _grouped["score_value"].sum(),
            "score_false": _grouped["score_miss"].sum(),
            "unparseable": _grouped["unparseable_row"].sum(),
            "score_rate": _grouped["score_value"].mean(),
        }
    )
    .assign(score_pct=lambda table: (table["score_rate"] * 100).round(1))
    .reindex(problem_type_order)
)
results_by_problem_type

,n,score_true,score_false,unparseable,score_rate,score_pct
problem_type,,,,,,
well_posed,90,15,60,15,0.166667,16.7
overlap_explicit,96,5,11,80,0.052083,5.2
overlap_implicit,48,2,6,40,0.041667,4.2
implausible,30,1,16,13,0.033333,3.3


In [148]:
df.columns

Index(['example_id', 'vignette_name', 'problem_type', 'intersection_size',
       'response_type', 'has_statistics', 'variant', 'prompt', 'well_posed',
       'normative', 'p_c_and_d_given_a', 'normative_choice',
       'normative_percent', 'normative_open', 'confidence_required',
       'numeric_score_percent', 'numeric_score_choice', 'scepticism_required',
       'scepticism_score_target', 'option_a_label', 'option_b_label',
       'option_c_label', 'option_d_label', 'option_e_label', 'option_a_lure',
       'option_b_lure', 'option_c_lure', 'option_d_lure', 'option_e_lure',
       'option_f_label', 'option_g_label', 'option_h_label', 'option_f_lure',
       'option_g_lure', 'option_h_lure', 'model', 'llm_response', 'reasoning',
       'answer_line', 'confidence_line', 'parsed_answer_type',
       'parsed_percent', 'parsed_choice', 'parsed_confidence', 'scoring_type',
       'parseable', 'score', 'score_value', 'parseable_bool', 'score_true'],
      dtype='str')

In [149]:
df['problem_type'].value_counts()

problem_type
overlap_explicit    96
well_posed          90
overlap_implicit    48
implausible         30
Name: count, dtype: int64

In [150]:
  df['variant'].value_counts()

variant
mc_full_probs       156
mc_numeric_probs     54
open_probs           54
Name: count, dtype: int64

In [151]:
 df['intersection_size'].value_counts()

intersection_size
0         120
large      72
medium     36
small      36
Name: count, dtype: int64

## `mc_numeric_probs` detail

For each vignette: MC options A–E, the model's letter (`parsed_choice`), partition shortcut letter (`numeric_score_choice`), scepticism fields, and score.

In [152]:
MC_NUMERIC_CHOICE_COLS = [f"option_{letter}_label" for letter in "abcde"]


def format_mc_choices(row: pd.Series) -> str:
    parts = []
    for letter, col in zip("ABCDE", MC_NUMERIC_CHOICE_COLS):
        value = row.get(col)
        if pd.notna(value) and str(value).strip():
            parts.append(f"{letter}: {value}")
    return " | ".join(parts)


mc_numeric_probs = df[df["variant"] == "mc_numeric_probs"].copy()
mc_numeric_probs["choices_offered"] = mc_numeric_probs.apply(format_mc_choices, axis=1)

mc_numeric_probs_view = mc_numeric_probs[
    [
        "vignette_name",
        "choices_offered",
        "parsed_choice",
        "numeric_score_choice",
        "scepticism_required",
        "scepticism_score_target",
        "score",
        "score_value",
        "normative_choice",
        "answer_line",
    ]
].sort_values("vignette_name")

pd.set_option("display.max_colwidth", 140)
mc_numeric_probs_view

,vignette_name,choices_offered,parsed_choice,numeric_score_choice,scepticism_required,scepticism_score_target,score,score_value,normative_choice,answer_line
12,CA Trump voter,A: About 6% | B: About 10% | C: About 1% | D: About 4% | E: About 13%,,B,false,n/a,false,0,A,- P(Trump|other CA) = 0.31
13,CA Trump voter,A: About 6% | B: About 10% | C: About 1% | D: About 4% | E: About 13%,A,B,false,n/a,true,1,A,A
14,CA Trump voter,A: About 6% | B: About 10% | C: About 1% | D: About 4% | E: About 13%,C,B,false,n/a,false,0,A,"- P(SC|C) = 0.60, P(OC|C) = 0.38"
15,CA Trump voter,A: About 6% | B: About 10% | C: About 1% | D: About 4% | E: About 13%,D,B,false,n/a,false,0,A,D
16,CA Trump voter,A: About 6% | B: About 10% | C: About 1% | D: About 4% | E: About 13%,D,B,false,n/a,false,0,A,D
17,CA Trump voter,A: About 6% | B: About 10% | C: About 1% | D: About 4% | E: About 13%,B,B,false,n/a,false,0,A,B
36,college STEM work,A: About 18% | B: About 20% | C: About 6% | D: About 3% | E: About 16%,,B,false,n/a,false,0,A,
37,college STEM work,A: About 18% | B: About 20% | C: About 6% | D: About 3% | E: About 16%,,B,false,n/a,false,0,A,
38,college STEM work,A: About 18% | B: About 20% | C: About 6% | D: About 3% | E: About 16%,,B,false,n/a,false,0,A,
39,college STEM work,A: About 18% | B: About 20% | C: About 6% | D: About 3% | E: About 16%,,B,false,n/a,false,0,A,


## `open_probs` detail

Benchmark scoring fields plus re-parsed response values. Each `llm_response` is re-parsed (strip trailing confidence, extract all % / 0–1 decimals). **`score_true`** is whether any candidate is within ±0.5 pp of **`scepticism_score_target`**.

In [153]:
import sys
from pathlib import Path

if "ROOT" not in globals():
    ROOT = Path.cwd()
    if not (ROOT / "data" / "base_rate").is_dir():
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.base_rate import (
    load_benchmark,
    matches_scepticism_target,
    parse_open_response,
    strip_trailing_confidence,
)

benchmark_items = load_benchmark()


def rescore_open_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_open_response(str(row["llm_response"]))
    body, confidence = strip_trailing_confidence(str(row["llm_response"]))
    score_true = matches_scepticism_target(item, parsed)
    return pd.Series(
        {
            "response_body": body,
            "parsed_confidence": confidence,
            "parsed_numbers": list(parsed.percent_candidates),
            "parsed_percent_rescored": parsed.percent,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.answer_type != "unparseable",
            "score_true": score_true,
        }
    )


def _csv_bool(series: pd.Series) -> pd.Series:
    return series.astype(bool).map(lambda value: "true" if value else "false")


open_probs = df[df["variant"] == "open_probs"].copy()
open_probs = open_probs.drop(columns=["score_true"], errors="ignore")
open_probs = pd.concat([open_probs, open_probs.apply(rescore_open_row, axis=1)], axis=1)

# Write rescored values back into the main frame for open_probs rows.
idx = open_probs.index
rescored_percent = pd.to_numeric(open_probs["parsed_percent_rescored"], errors="coerce")
df.loc[idx, "parsed_percent"] = rescored_percent.map(
    lambda value: "" if pd.isna(value) else f"{value:g}"
)
df.loc[idx, "parsed_answer_type"] = open_probs["parsed_answer_type_rescored"].astype("string")
df.loc[idx, "parseable"] = _csv_bool(open_probs["parseable_rescored"])
df.loc[idx, "parseable_bool"] = open_probs["parseable_rescored"].astype(bool)
df.loc[idx, "score"] = _csv_bool(open_probs["score_true"])
df.loc[idx, "score_value"] = open_probs["score_true"].astype(int)
df.loc[idx, "score_true"] = open_probs["score_true"].astype(bool)

OPEN_PROBS_SCORING_COLUMNS = [
    "example_id",
    "vignette_name",
    "normative",
    "scepticism_required",
    "normative_percent",
    "normative_open",
    "numeric_score_percent",
    "scepticism_score_target",
    "parsed_numbers",
    "parsed_percent_rescored",
    "parsed_answer_type_rescored",
    "parsed_confidence",
    "parseable_rescored",
    "score_true",
]

open_probs_view = (
    open_probs[OPEN_PROBS_SCORING_COLUMNS]
    .sort_values(["vignette_name", "normative"])
    .reset_index(drop=True)
)

print(
    "Rescored open_probs score_true:",
    int(open_probs["score_true"].sum()),
    "/",
    len(open_probs),
)
pd.set_option("display.max_colwidth", 120)
open_probs_view

TypeError: Invalid value for dtype 'str'. Value should be a string or missing value (or array of those).

## `open_probs` vs `mc_numeric_probs` vs normative

Side-by-side for the **7 canonical** vignettes (both variants present). **Normative** = overlap-aware P(A|T) (`normative_percent`). **Partition** = partition-shortcut P(A|T) (`numeric_score_percent`; MC numeric score target is `n/a`, pass = normative letter or same rounded %).

In [ ]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.base_rate import load_benchmark, parse_open_response, parse_response

benchmark_items = load_benchmark()
items_meta = pd.read_csv(ROOT / "data" / "base_rate" / "items.csv")


def _label_percent(label: str) -> float | None:
    text = (label or "").strip()
    if not text.startswith("About "):
        return None
    try:
        return float(text.removeprefix("About ").removesuffix("%"))
    except ValueError:
        return None


def _format_mc_menu(row: pd.Series) -> str:
    parts = []
    for letter in "ABCDE":
        label = row.get(f"option_{letter}_label")
        if pd.notna(label) and str(label).strip():
            parts.append(f"{letter}:{label}")
    return " | ".join(parts)


comparison_rows: list[dict] = []
canonical_vignettes = sorted(
    items_meta.loc[
        ~items_meta["scepticism_required"].astype(str).str.lower().eq("true"),
        "vignette_name",
    ].unique()
)

for vignette_name in canonical_vignettes:
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = item_mc.get(f"option_{mc_choice.lower()}_label", "") if mc_choice else ""
    mc_pct = _label_percent(str(mc_label))

    normative_pct = float(item_open["normative_percent"])
    partition_pct = float(item_mc["numeric_score_percent"])
    open_pct = parsed_open.percent

    comparison_rows.append(
        {
            "vignette_name": vignette_name,
            "normative_open": item_open["normative_open"],
            "partition_pct": partition_pct,
            "normative_mc_letter": item_mc["normative_choice"],
            "mc_menu": _format_mc_menu(item_mc),
            "open_parsed_pct": open_pct,
            "open_score": bool(open_row.get("score_true", open_row.get("score_value", 0))),
            "mc_choice": mc_choice,
            "mc_label": mc_label,
            "mc_parsed_pct": mc_pct,
            "mc_delta_vs_partition_pp": None if mc_pct is None else mc_pct - partition_pct,
            "mc_score": bool(mc_row.get("score_true", mc_row.get("score_value", 0))),
            "same_norm_partition_rounded": round(normative_pct) == round(partition_pct),
        }
    )

open_vs_mc = pd.DataFrame(comparison_rows).sort_values("vignette_name")

print(
    "open_probs pass:",
    int(open_vs_mc["open_score"].sum()),
    "/",
    len(open_vs_mc),
    "| mc_numeric_probs pass:",
    int(open_vs_mc["mc_score"].sum()),
    "/",
    len(open_vs_mc),
)

display(
    open_vs_mc[
        [
            "vignette_name",
            "normative_open",
            "partition_pct",
            "open_parsed_pct",
            "open_score",
            "mc_choice",
            "mc_label",
            "mc_parsed_pct",
            "mc_score",
            "normative_mc_letter",
        ]
    ]
)

with pd.option_context("display.max_colwidth", None):
    display(open_vs_mc[["vignette_name", "mc_menu"]])

### Printable comparison (canonical vignettes)

Per vignette: source probabilities (vignette CSVs via `load_vignettes`; `items.csv` stores `p_c_and_d_given_a` only), normative / open / MC answers, numeric MC options, and full `mc_numeric_probs` prompt.

In [ ]:
import re

from benchmarks.base_rate import parse_open_response, parse_response
from scripts.build_base_rate_prompts import load_vignettes

items_meta = pd.read_csv(ROOT / "data" / "base_rate" / "items.csv")
benchmark_df = pd.read_csv(ROOT / "data" / "base_rate" / "benchmark.csv")

canonical_vignettes = sorted(
    items_meta.loc[
        ~items_meta["scepticism_required"].astype(str).str.lower().eq("true"),
        "vignette_name",
    ].unique()
)

vignette_by_name = {
    v.name: v
    for v in load_vignettes()
    if v.normative != "implausible"
}


def mc_numeric_options_prompt(prompt: str) -> str:
    """Extract the A–E option block from an mc_numeric_probs prompt."""
    lines = [line.strip() for line in prompt.splitlines() if line.strip()]
    option_lines = [line for line in lines if re.match(r"^[A-E]\.\s", line)]
    return " | ".join(option_lines)


def format_source_ps(v, item_row: pd.Series) -> str:
    parts = [
        f"P(A)={v.p_a:.6g}",
        f"P(C|A)={v.q_c:.6g}",
        f"P(D|A)={v.q_d:.6g}",
        f"P(T|C)={v.s_c:.6g}",
        f"P(T|D)={v.s_d:.6g}",
        f"P(T|N)={v.f_n:.6g}",
    ]
    p_cd_items = item_row.get("p_c_and_d_given_a")
    if pd.notna(p_cd_items):
        parts.append(f"P(C∩D|A)={float(p_cd_items):.6g} (items.csv)")
    elif v.p_cd:
        parts.append(f"P(C∩D|A)={v.p_cd:.6g}")
    return " | ".join(parts)


print_rows: list[dict] = []
for vignette_name in canonical_vignettes:
    open_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "open_probs")].iloc[0]
    mc_row = df[(df["vignette_name"] == vignette_name) & (df["variant"] == "mc_numeric_probs")].iloc[0]
    item_open = items_meta.loc[items_meta["example_id"] == open_row["example_id"]].iloc[0]
    item_mc = items_meta.loc[items_meta["example_id"] == mc_row["example_id"]].iloc[0]
    bench_row = benchmark_df.loc[benchmark_df["example_id"] == mc_row["example_id"]].iloc[0]
    vignette = vignette_by_name[vignette_name]

    parsed_open = parse_open_response(str(open_row["llm_response"]))
    parsed_mc = parse_response(str(mc_row["llm_response"]), scoring_type="mc_numeric")
    mc_choice = parsed_mc.choice or ""
    mc_label = (
        str(item_mc.get(f"option_{mc_choice.lower()}_label", ""))
        if mc_choice
        else ""
    )
    full_prompt = str(bench_row["prompt"])

    print_rows.append(
        {
            "vignette_name": vignette_name,
            "source_ps": format_source_ps(vignette, item_open),
            "p_a": vignette.p_a,
            "p_c_given_a": vignette.q_c,
            "p_d_given_a": vignette.q_d,
            "p_t_given_c": vignette.s_c,
            "p_t_given_d": vignette.s_d,
            "p_t_given_n": vignette.f_n,
            "p_c_and_d_given_a": float(item_open["p_c_and_d_given_a"]),
            "normative_pct": float(item_open["normative_percent"]),
            "open_parsed_pct": parsed_open.percent,
            "mc_label": f"{mc_choice} {mc_label}".strip(),
            "numeric_prompt": mc_numeric_options_prompt(full_prompt),
            "prompt": full_prompt,
        }
    )

print_table = pd.DataFrame(print_rows).sort_values("vignette_name")

print(f"{'vignette_name':<32} {'normative':>10} {'open':>10} {'MC label':>14}")
print("-" * 72)
for row in print_table.itertuples(index=False):
    open_pct = "—" if pd.isna(row.open_parsed_pct) else f"{row.open_parsed_pct:.4g}%"
    print(f"\n{row.vignette_name}")
    print(f"  source Ps:   {row.source_ps}")
    print(f"  normative:   {row.normative_pct:.4g}%")
    print(f"  open parsed: {open_pct}")
    print(f"  MC label:    {row.mc_label}")
    print(f"  numeric prompt: {row.numeric_prompt}")
    print("  prompt:")
    for line in row.prompt.splitlines():
        print(f"    {line}")

print_table.drop(columns=["prompt"])

In [ ]:
open_probs_responses = (
    open_probs[
        ["vignette_name", "normative_open", "parsed_numbers", "llm_response"]
    ]
    .sort_values("vignette_name")
    .reset_index(drop=True)
)

with pd.option_context("display.max_colwidth", None, "display.width", None):
    display(open_probs_responses)

## `mc_full_probs` detail

Benchmark scoring fields plus re-parsed MC choice (bottom-up scan for A–H), **only rows with `scepticism_required=true`**. **`score`** is whether **`parsed_choice_rescored`** matches **`scepticism_score_target`** (single letter, or any of `F|G|H`).

In [ ]:
from benchmarks.base_rate import (
    load_benchmark,
    matches_scepticism_target,
    parse_response,
)

if "benchmark_items" not in globals():
    benchmark_items = load_benchmark()


def rescore_mc_full_row(row: pd.Series) -> pd.Series:
    item = benchmark_items[row["example_id"]]
    parsed = parse_response(str(row["llm_response"]), scoring_type="mc_full")
    score_true = matches_scepticism_target(item, parsed)
    return pd.Series(
        {
            "answer_line_rescored": parsed.answer_line,
            "parsed_confidence": parsed.confidence,
            "parsed_choice_rescored": parsed.choice,
            "parsed_answer_type_rescored": parsed.answer_type,
            "parseable_rescored": parsed.choice is not None,
            "score": score_true,
        }
    )


mc_full_probs = df[df["variant"] == "mc_full_probs"].copy()
mc_full_probs = mc_full_probs.drop(columns=["score"], errors="ignore")
mc_full_probs = pd.concat(
    [mc_full_probs, mc_full_probs.apply(rescore_mc_full_row, axis=1)],
    axis=1,
)

MC_FULL_PROBS_SCORING_COLUMNS = [
    "example_id",
    "vignette_name",
    "normative",
    "scepticism_required",
    "normative_choice",
    "numeric_score_choice",
    "scepticism_score_target",
    "parsed_choice_rescored",
    "parsed_answer_type_rescored",
    "answer_line_rescored",
    "parsed_confidence",
    "parseable_rescored",
    "score",
]

mc_full_probs_view = (
    mc_full_probs.loc[
        mc_full_probs["scepticism_required"].astype(str).str.lower().eq("true"),
        MC_FULL_PROBS_SCORING_COLUMNS,
    ]
    .sort_values(["vignette_name", "normative"])
    .reset_index(drop=True)
)

print(
    "mc_full_probs score (scepticism_required):",
    int(mc_full_probs_view["score"].sum()),
    "/",
    len(mc_full_probs_view),
)

pd.set_option("display.max_colwidth", 120)
mc_full_probs_view.head(n=5)

In [ ]:
df['reasoning'].value_counts(), df['confidence_required'].value_counts(), df['normative_choice'].value_counts()

## Scores by `response_type`

In [ ]:
RESPONSE_TYPE_ORDER = ["open", "mc_numeric", "mc_full"]


def score_summary_table(group_col: str, *, order: list[str] | None = None) -> pd.DataFrame:
    """Counts, parseability mix, and mean score for each group value."""
    work = df.copy()
    if "parseable_bool" not in work.columns:
        work["parseable_bool"] = True
    work["score_miss"] = work["parseable_bool"] & (work["score_value"] == 0)
    work["unparseable_row"] = ~work["parseable_bool"]

    grouped = work.groupby(group_col, observed=True)
    summary = pd.DataFrame(
        {
            "n": grouped.size(),
            "score_true": grouped["score_value"].sum(),
            "score_false": grouped["score_miss"].sum(),
            "unparseable": grouped["unparseable_row"].sum(),
            "score_rate": grouped["score_value"].mean(),
        }
    )
    summary["score_pct"] = (summary["score_rate"] * 100).round(1)

    if order:
        summary = summary.reindex([value for value in order if value in summary.index])

    return summary


by_response_type = score_summary_table("response_type", order=RESPONSE_TYPE_ORDER)
by_response_type

## Scores by `variant`

In [ ]:
VARIANT_ORDER = [
    "open_probs",
    "mc_numeric_probs",
    "mc_full_probs",
]

by_variant = score_summary_table("variant", order=VARIANT_ORDER)
by_variant

## Scores by `vignette_name`

In [ ]:
by_vignette = score_summary_table(
    "vignette_name",
    order=sorted(df["vignette_name"].unique()),
)
by_vignette

## Optional: split by model when multiple LLMs are present

In [ ]:
if df["model"].nunique() > 1:
    display(
        df.groupby(["model", "response_type"], observed=True)["score_value"]
        .mean()
        .unstack("response_type")
        .reindex(columns=RESPONSE_TYPE_ORDER)
        .round(3)
    )
    display(
        df.groupby(["model", "variant"], observed=True)["score_value"]
        .mean()
        .unstack("variant")
        .reindex(columns=VARIANT_ORDER)
        .round(3)
    )
else:
    print("Single model in file — see tables above.")

In [ ]:
df['model'].value_counts()

In [ ]:
Trump = df.query("vignette_name == 'CA Trump voter'")
Trump